In [38]:
import pandas as pd

drivers = pd.read_csv("clean_drivers.csv")
trips = pd.read_csv("trips.csv")
goals = pd.read_csv("driver_goals.csv")
velocity = pd.read_csv("earnings_velocity_log.csv")

In [39]:
trips["earnings"] = trips["fare"] * trips["surge_multiplier"]

In [41]:
trips = trips.rename(columns={
    "start_time":"trip_start_time",
    "end_time":"trip_end_time"
})

In [42]:
trips = trips.sort_values(["driver_id","trip_start_time"])

In [43]:
trips["cum_earnings"] = trips.groupby("driver_id")["earnings"].cumsum()

trips["trip_number"] = trips.groupby("driver_id").cumcount()+1

In [44]:
trips["trip_start_time"] = pd.to_datetime(trips["trip_start_time"])
trips["trip_end_time"] = pd.to_datetime(trips["trip_end_time"])

trips["trip_duration"] = (
    trips["trip_end_time"] - trips["trip_start_time"]
).dt.total_seconds()/3600

/var/folders/_p/hqzp429958z7x728kp98t6ym0000gn/T/ipykernel_26949/3363310791.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  trips["trip_start_time"] = pd.to_datetime(trips["trip_start_time"])
/var/folders/_p/hqzp429958z7x728kp98t6ym0000gn/T/ipykernel_26949/3363310791.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  trips["trip_end_time"] = pd.to_datetime(trips["trip_end_time"])


In [45]:
trips["cum_hours"] = trips.groupby("driver_id")["trip_duration"].cumsum()

In [46]:
trips["earnings_velocity"] = trips["cum_earnings"] / trips["cum_hours"]

In [47]:
data = trips.merge(goals, on="driver_id")

In [48]:
data["remaining_target"] = data["target_earnings"] - data["cum_earnings"]

In [49]:
data = data.merge(goals, on="driver_id")

In [50]:
print(data.columns)

Index(['trip_id', 'driver_id', 'date_x', 'trip_start_time', 'trip_end_time',
       'duration_min', 'distance_km', 'fare', 'surge_multiplier',
       'pickup_location', 'dropoff_location', 'trip_status', 'earnings',
       'cum_earnings', 'trip_number', 'trip_duration', 'cum_hours',
       'earnings_velocity_x', 'goal_id_x', 'date_y', 'shift_start_time_x',
       'shift_end_time_x', 'target_earnings_x', 'target_hours_x',
       'current_earnings_x', 'current_hours_x', 'status_x',
       'earnings_velocity_y', 'goal_completion_forecast_x', 'remaining_target',
       'goal_id_y', 'date', 'shift_start_time_y', 'shift_end_time_y',
       'target_earnings_y', 'target_hours_y', 'current_earnings_y',
       'current_hours_y', 'status_y', 'earnings_velocity',
       'goal_completion_forecast_y'],
      dtype='object')


In [52]:
data = data.rename(columns={
    "shift_start_time_y": "shift_start_time",
    "shift_end_time_y": "shift_end_time",
    "target_earnings_y": "target_earnings"
})
data = data.drop(columns=[
    "shift_start_time_x",
    "shift_end_time_x",
    "target_earnings_x"
])

In [53]:
data["shift_start_time"] = pd.to_datetime(data["shift_start_time"])
data["shift_end_time"] = pd.to_datetime(data["shift_end_time"])
data["trip_end_time"] = pd.to_datetime(data["trip_end_time"])
data["trip_start_time"] = pd.to_datetime(data["trip_start_time"])

/var/folders/_p/hqzp429958z7x728kp98t6ym0000gn/T/ipykernel_26949/3784074624.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["shift_start_time"] = pd.to_datetime(data["shift_start_time"])
/var/folders/_p/hqzp429958z7x728kp98t6ym0000gn/T/ipykernel_26949/3784074624.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["shift_end_time"] = pd.to_datetime(data["shift_end_time"])


In [54]:
data["total_shift_hours"] = (
    data["shift_end_time"] - data["shift_start_time"]
).dt.total_seconds() / 3600

In [55]:
data["remaining_hours"] = (
    data["shift_end_time"] - data["trip_end_time"]
).dt.total_seconds() / 3600

In [56]:
data["remaining_hours"] = data["remaining_hours"].clip(lower=0)

In [69]:
data["shift_progress"] = (
    data["elapsed_shift_hours"] / data["total_shift_hours"]
)

In [58]:
data["elapsed_shift_hours"] = (
    data["trip_end_time"] - data["shift_start_time"]
).dt.total_seconds() / 3600

In [59]:
data["required_velocity"] = (
    data["remaining_target"] / data["remaining_hours"]
)

In [60]:
data["fare_per_km"] = data["fare"] / data["distance_km"]

In [61]:
data["goal_completion_ratio"] = data["cum_earnings"] / data["target_earnings"]

In [62]:
data["time_remaining_ratio"] = data["remaining_hours"] / data["total_shift_hours"]

In [63]:
data["trips_per_hour"] = data["trip_number"] / data["cum_hours"]

In [64]:
data["earnings_per_hour"] = data["cum_earnings"] / data["cum_hours"]

In [65]:
data["fare_per_hour"] = data["earnings"] / data["trip_duration"]

In [66]:
driver_avg_velocity = data.groupby("driver_id")["earnings_velocity"].mean()

data["driver_avg_velocity"] = data["driver_id"].map(driver_avg_velocity)

In [67]:
data["earnings_last3"] = (
    data.groupby("driver_id")["earnings"]
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)

In [68]:
data["momentum_score"] = data["earnings_last3"] - data["driver_avg_velocity"]

In [70]:
data["speed_kmph"] = data["distance_km"] / data["trip_duration"]

In [71]:
data["hour_of_day"] = data["trip_end_time"].dt.hour

In [72]:
data["is_peak_hour"] = data["hour_of_day"].isin([8,9,10,17,18,19]).astype(int)

In [73]:
data["surge_last3"] = (
    data.groupby("driver_id")["surge_multiplier"]
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)

In [74]:
location_demand = data["pickup_location"].value_counts()

data["pickup_demand"] = data["pickup_location"].map(location_demand)

In [75]:
drop_demand = data["dropoff_location"].value_counts()

data["dropoff_demand"] = data["dropoff_location"].map(drop_demand)

In [77]:
import numpy as np

data = data.replace([np.inf, -np.inf], 0)
data = data.fillna(0)